# SIPTA — Ingesta y EDA: Ambiente, Calidad del Aire y Conflictos SAC
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona C (Sofía Hidalgo — Ingesta & EDA)**  
**Colaboración / Integración**: Persona A (Adan Sánchez — Lead Data Engineer)  
**Objetivo**: Ingesta y análisis exploratorio (EDA) de estaciones RMCAB y Situaciones Ambientales Conflictivas (SAC).  
**Datos de Entrada**: `data/raw/AMBIENTE/*`  
**Datos de Salida**: `data/processed/AMBIENTE/*`


## 1. Ingesta y Análisis Exploratorio de Datos Ambientales



# SIPTA Notebook: Ingestión de datos
Este notebook sirve como guía inicial para la ingesta de datos crudos en el proyecto SIPTA.

### Objetivos
- Registrar y versionar las fuentes de datos originales.
- Cargar archivos desde `data/raw` con pandas.
- Guardar copias de seguridad para reproducibilidad.

In [4]:
from pathlib import Path
import logging
import json
import pandas as pd
import requests

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio Raw: {RAW_DIR}")
print(f"Directorio Processed: {PROCESSED_DIR}")

ModuleNotFoundError: No module named 'requests'

## 1. Funciones de Carga Reutilizables

In [ ]:
def load_raw_csv(filename: str, subcarpeta: str = "", separador: str = ';', encoding: str = 'utf-8') -> pd.DataFrame:
    """Carga un dataset crudo en formato CSV."""
    path = RAW_DIR / subcarpeta / filename if subcarpeta else RAW_DIR / filename
    assert path.exists(), f'No existe el archivo en la ruta: {path}'
    return pd.read_csv(path, sep=separador, low_memory=False, encoding=encoding)

def load_raw_geojson_points(filename: str, subcarpeta: str = "") -> pd.DataFrame:
    """Carga un GeoJSON crudo de puntos y extrae propiedades y coordenadas a DataFrame."""
    path = RAW_DIR / subcarpeta / filename if subcarpeta else RAW_DIR / filename
    assert path.exists(), f'No existe el archivo GeoJSON: {path}'
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    records = []
    for feat in data.get("features", []):
        props = feat.get("properties", {}).copy()
        geometry = feat.get("geometry", {})
        if geometry and geometry.get("type") == "Point":
            coords = geometry.get("coordinates", [None, None])
            props["lon"] = coords[0]
            props["lat"] = coords[1]
        records.append(props)
    return pd.DataFrame(records)

def save_versioned_raw(df: pd.DataFrame, filename: str, subcarpeta: str = "", suffix: str = 'v1') -> Path:
    """Guarda versión inmutable en data/raw."""
    directorio = RAW_DIR / subcarpeta if subcarpeta else RAW_DIR
    directorio.mkdir(parents=True, exist_ok=True)
    output = directorio / f"{Path(filename).stem}_{suffix}.csv"
    df.to_csv(output, index=False, encoding='utf-8')
    logging.info(f"Copia versionada guardada en: {output}")
    return output

## 2. Ingesta: Dominio Ambiente (D5)

In [ ]:
# 1. Situaciones Ambientales Conflictivas (SAC)
df_sac = load_raw_csv('situacion_ambiental_conflictiva.csv', subcarpeta='AMBIENTE', separador=';', encoding='latin1')

# 2. Estaciones de Calidad del Aire (RMCAB)
df_aire = load_raw_geojson_points('estacion_calidad_aire.geojson', subcarpeta='AMBIENTE')

print(f"Dataset SAC cargado: {df_sac.shape[0]} filas, {df_sac.shape[1]} columnas.")
print(f"Dataset Calidad del Aire cargado: {df_aire.shape[0]} estaciones.")

In [ ]:
print("=== 5 PRIMERAS FILAS: SAC ===")
display(df_sac.head())

print("=== 5 PRIMERAS FILAS: ESTACIONES AIRE ===")
display(df_aire.head())

In [ ]:
print("=== REVISIÓN TERRITORIAL: SAC ===")
print("Localidades únicas:", sorted(df_sac["localidad"].dropna().astype(str).unique()))
print("Códigos únicos:", sorted(df_sac["cod_locali"].dropna().astype(str).unique()))

print("\n=== REVISIÓN TERRITORIAL: CALIDAD DEL AIRE ===")
print("Códigos de localidad cubiertos:", sorted(df_aire["sect_loc"].dropna().unique()))

In [ ]:
print("=== ANÁLISIS DE NULOS: SAC ===")
nulos_sac = df_sac.isna().sum()
pct_sac = (df_sac.isna().mean() * 100).round(2)
display(pd.DataFrame({'nulos': nulos_sac, 'porcentaje (%)': pct_sac}).sort_values('porcentaje (%)', ascending=False))

print(f"\nFilas duplicadas en SAC: {df_sac.duplicated().sum()}")
print(f"Filas duplicadas en Estaciones Aire: {df_aire.duplicated().sum()}")

In [ ]:
# Guardar respaldos versionados
save_versioned_raw(df_sac, 'situacion_ambiental_conflictiva.csv', subcarpeta='AMBIENTE', suffix='20260731')
save_versioned_raw(df_aire, 'estacion_calidad_aire.csv', subcarpeta='AMBIENTE', suffix='20260731')

### Notas de Ingesta — Ambiente (D5)
- **Origen:** Datos crudos ubicados en `data/raw/AMBIENTE/`.
- **Parámetros:** SAC usa separador `;` y codificación `latin1`. Estaciones de aire aplanadas desde GeoJSON.
- **⚠ Pendiente de validar con datos:** Desplazamiento de columnas en ~30 filas de SAC por saltos de línea sin entrecomillar.
- **⚠ Pendiente de validar con datos:** Formato decimal con puntos como miles en coordenadas que se corregirá en limpieza.